In [1]:
a = 10

In [2]:
print(a)

10


In [3]:
a += 5
a

15

In [4]:
import torch
import torch.nn as nn
import torch.nn.functional as F
import torch.optim as optim
from torch.utils.data import DataLoader, TensorDataset

from sklearn.datasets import load_breast_cancer
from sklearn.model_selection import train_test_split
from sklearn.preprocessing import StandardScaler

In [5]:
X, y = load_breast_cancer(return_X_y=True)
X_train, X_test, y_train, y_test = train_test_split(X, y, test_size=0.2)

scaler = StandardScaler()

X_train_scaled = scaler.fit_transform(X_train)
X_test_scaled = scaler.transform(X_test)

In [6]:
X_train_scaled

array([[ 0.52265434, -1.05616967,  0.4621283 , ..., -0.04374998,
        -0.2176069 , -0.75950747],
       [-0.55958539, -1.23793603, -0.55700042, ..., -0.36331259,
        -0.3787459 ,  0.25558893],
       [-0.81844171, -1.50468406, -0.78143237, ...,  0.23716786,
         0.68381424,  0.47463015],
       ...,
       [ 0.23075253,  1.51216535,  0.21054089, ...,  0.54632611,
        -0.41544092,  0.59955647],
       [-0.39435795, -1.48107804, -0.45676479, ..., -1.12877654,
        -1.80506538, -0.5875237 ],
       [-0.33928214,  0.68123343, -0.42801194, ..., -1.58420529,
        -0.98820232, -1.4143903 ]], shape=(455, 30))

In [7]:
y_train

array([1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 0, 0, 1, 0, 0, 0,
       0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 1, 1, 0,
       1, 0, 1, 1, 1, 1, 1, 0, 1, 1, 0, 0, 1, 0, 1, 0, 1, 1, 1, 1, 1, 1,
       1, 1, 0, 1, 1, 0, 0, 0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 0, 1, 1, 0, 0,
       1, 1, 0, 1, 0, 0, 1, 1, 0, 1, 1, 0, 1, 0, 0, 0, 0, 0, 1, 1, 0, 1,
       1, 1, 0, 1, 0, 0, 0, 1, 1, 1, 1, 0, 1, 0, 1, 1, 0, 1, 1, 1, 1, 0,
       1, 1, 0, 1, 0, 1, 1, 1, 0, 0, 0, 1, 0, 1, 0, 0, 0, 1, 0, 1, 1, 1,
       1, 1, 1, 0, 1, 0, 1, 0, 1, 0, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1, 0, 0,
       0, 1, 0, 1, 0, 0, 0, 1, 0, 0, 1, 0, 1, 1, 1, 1, 0, 1, 1, 1, 0, 0,
       0, 1, 1, 1, 0, 0, 1, 0, 0, 1, 0, 1, 1, 0, 1, 0, 0, 1, 1, 1, 1, 1,
       0, 1, 1, 1, 1, 1, 0, 0, 0, 1, 1, 1, 0, 1, 1, 1, 1, 0, 1, 1, 0, 1,
       1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 0, 0, 0, 0, 1, 0, 1, 1, 0, 1, 0,
       1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 0, 1, 1, 1, 1, 1, 0, 1, 1, 1,
       0, 1, 1, 1, 0, 0, 0, 0, 1, 1, 1, 1, 1, 1, 0,

In [8]:
X_train_scaled_tensor = torch.from_numpy(X_train_scaled).float()
X_test_scaled_tensor = torch.from_numpy(X_test_scaled).float()

y_train_tensor = torch.from_numpy(y_train).unsqueeze(1).float()
y_test_tensor = torch.from_numpy(y_test).unsqueeze(1).float()

In [9]:
train_dataset = TensorDataset(X_train_scaled_tensor, y_train_tensor)

In [10]:
X_train_scaled_tensor.shape

torch.Size([455, 30])

In [11]:
train_loader = DataLoader(train_dataset, batch_size=32, shuffle=True)

In [12]:
class BCNet(nn.Module):
    def __init__(self):
        super(BCNet, self).__init__()

        self.fc1 = nn.Linear(30, 64)
        self.fc2 = nn.Linear(64, 32)
        self.fc3 = nn.Linear(32, 1)

        self.dropout = nn.Dropout(0.5)

    def forward(self, x):
        x = F.relu(self.fc1(x))
        x = F.relu(self.fc2(x))
        x = self.dropout(x)
        x = F.sigmoid(self.fc3(x))

        return x

In [13]:
model = BCNet()

In [14]:
criterion = nn.BCELoss()
optimizer = optim.Adam(model.parameters(), lr=0.001)

In [15]:
epochs = 20

for epoch in range(epochs):
    model.train()
    running_loss = 0.0

    for x_batch, y_batch in train_loader:
        optimizer.zero_grad()

        preds = model(x_batch)
        loss = criterion(preds, y_batch)

        loss.backward()
        optimizer.step()

        running_loss += loss.item()

    print(f'Epoch {epoch+1}: Loss was {running_loss / len(train_loader)}')
        

Epoch 1: Loss was 0.6581623792648316
Epoch 2: Loss was 0.5540570040543874
Epoch 3: Loss was 0.41002981464068095
Epoch 4: Loss was 0.262386683622996
Epoch 5: Loss was 0.170346071322759
Epoch 6: Loss was 0.12323559286693732
Epoch 7: Loss was 0.10095469082395235
Epoch 8: Loss was 0.09602939163645109
Epoch 9: Loss was 0.08375619128346443
Epoch 10: Loss was 0.08662175560990969
Epoch 11: Loss was 0.08223334985474745
Epoch 12: Loss was 0.05846867534952859
Epoch 13: Loss was 0.06381622535021354
Epoch 14: Loss was 0.05881254921356837
Epoch 15: Loss was 0.05688809069494406
Epoch 16: Loss was 0.055535505246371034
Epoch 17: Loss was 0.04461673672000567
Epoch 18: Loss was 0.04531238625446955
Epoch 19: Loss was 0.049662827824552856
Epoch 20: Loss was 0.04947212655097246


In [16]:
with torch.no_grad():
    model.eval()

    preds = model(X_test_scaled_tensor)
    loss = criterion(preds, y_test_tensor).item()

    accuracy = ((preds >= 0.5) == y_test_tensor).float().mean().item()

In [17]:
accuracy

0.9736841917037964